# Flowchart/Pseudocode → Java Alignment Verifier (Kaggle T4×2)
Thin entry point. Loads Qwen2.5-VL-7B offline and writes `submission.csv`.
Set the two paths below to your attached **Model** and **competition data**.

In [ ]:
import os, sys
# --- point these at your Kaggle inputs -----------------------------------
os.environ['ALIGN_MODEL_DIR'] = '/kaggle/input/qwen2.5-vl-7b-instruct/transformers/default/1'
os.environ['ALIGN_DATA_DIR']  = '/kaggle/input/flowchart-to-code'
os.environ['ALIGN_OUTPUT_DIR'] = '/kaggle/working'
# code modules: attach this repo as a dataset, or clone it here
CODE_DIR = '/kaggle/input/alignment-verifier-code'  # or '.' if pasted in the notebook
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
os.environ['HF_HUB_OFFLINE'] = '1'; os.environ['TRANSFORMERS_OFFLINE'] = '1'

In [ ]:
# Install the vision helper if the image lacks it (from an attached wheel dataset when offline)
!pip install -q qwen-vl-utils 2>/dev/null || echo 'qwen-vl-utils already present or offline'

In [ ]:
import importlib, config
importlib.reload(config)
import predict
importlib.reload(predict)

fewshot = predict.build_fewshot()
judge = predict.Judge()   # loads Qwen2.5-VL-7B (fp16 / sdpa, or 4-bit if configured)

In [ ]:
# Sanity: run the first 3 rows verbose before the full pass
_ = predict.run_submission(judge, fewshot, dry_run=3)

In [ ]:
# Optional: exact-match accuracy + confusion matrix on the labeled train set
predict.run_validation(judge, fewshot)

In [ ]:
# Full submission
sub = predict.run_submission(judge, fewshot)
config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sub.to_csv(config.SUBMISSION_PATH, index=False)
print('wrote', len(sub), 'rows ->', config.SUBMISSION_PATH)
sub.head()